In [1]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cosine
import os

In [ ]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/metadata/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

def metadata_path_for_video(video_path):
    video_stem = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(
        "/home/guilherme_monteiro/projetos/tcc/data/metadata",
        f"{video_stem}_meta.json"
    )

df['metadata_path'] = df['video_path'].apply(metadata_path_for_video)
df['has_metadata'] = df['metadata_path'].apply(os.path.exists)
df_meta = df[df['has_metadata']].reset_index(drop=True)

print(f"Vídeos no CSV: {len(df)}")
print(f"Vídeos com metadata disponível: {len(df_meta)}")
print(df_meta['Video Ground Truth'].value_counts())


# Funções básicas

In [ ]:
def get_video_frame_count(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frame_count


def sample_frame_indices(frame_count, max_frames=None):
    if frame_count <= 0:
        return np.array([], dtype=int)

    if max_frames is None or frame_count <= max_frames:
        return np.arange(frame_count, dtype=int)

    return np.linspace(0, frame_count - 1, int(max_frames)).astype(int)


def load_video_frames(video_path, max_frames=None, return_indices=False):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_frame_indices(frame_count, max_frames)

    frames = []
    used_indices = []

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()

        if not ret:
            continue

        frames.append(frame)
        used_indices.append(int(frame_idx))

    cap.release()

    frames = np.array(frames)

    if return_indices:
        return frames, np.array(used_indices, dtype=int), frame_count

    return frames


def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale


def load_metadata(video_path):
    full_path = metadata_path_for_video(video_path)

    with open(full_path, "r") as f:
        return json.load(f)


def metadata_index_for_frame(frame_idx, frame_count, metadata_len):
    if metadata_len <= 0:
        return None

    if metadata_len >= frame_count - 3:
        return min(int(frame_idx), metadata_len - 1)

    metadata_frame_indices = sample_frame_indices(frame_count, metadata_len)
    return int(np.argmin(np.abs(metadata_frame_indices - int(frame_idx))))


def scale_bbox(bbox, scale):
    return [int(round(x * scale)) for x in bbox]


def clip_bbox(bbox, width, height, min_size=4):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]

    x1 = max(0, min(width - 1, x1))
    x2 = max(0, min(width, x2))
    y1 = max(0, min(height - 1, y1))
    y2 = max(0, min(height, y2))

    if x2 <= x1:
        x2 = min(width, x1 + min_size)
    if y2 <= y1:
        y2 = min(height, y1 + min_size)

    return [x1, y1, x2, y2]


def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = clip_bbox(bbox, w, h)

    bw = x2 - x1
    bh = y2 - y1

    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    border_mask = expanded_mask - face_mask
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox": (x1, y1, x2, y2),
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Função para visualização

In [4]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# LBP

In [5]:
def compute_lbp(gray, P=8, R=1):
    lbp = local_binary_pattern(gray, P, R, method="uniform")
    return lbp

In [ ]:
def lbp_region_features(lbp, mask, n_bins=10):
    values = lbp[mask == 1]

    if len(values) < 10:
        return None

    hist, _ = np.histogram(values, bins=n_bins, range=(0, n_bins), density=False)
    hist = hist.astype(np.float64)
    hist /= (hist.sum() + 1e-12)

    entropy = float(-np.sum(hist * np.log(hist + 1e-12)) / np.log(n_bins))
    uniformity = float(np.sum(hist ** 2))
    sparsity = float(np.mean(hist > 0.01))

    return {
        "hist": hist,
        "entropy_norm": entropy,
        "uniformity": uniformity,
        "sparsity": sparsity,
    }


In [ ]:
def safe_cosine_hist(h1, h2):
    if h1 is None or h2 is None:
        return np.nan

    if np.linalg.norm(h1) < 1e-12 or np.linalg.norm(h2) < 1e-12:
        return 1.0

    return float(cosine(h1, h2))


def lbp_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_lbp_entropy_norm_diff": abs(f1["entropy_norm"] - f2["entropy_norm"]),
        f"{prefix}_lbp_uniformity_diff": abs(f1["uniformity"] - f2["uniformity"]),
        f"{prefix}_lbp_sparsity_diff": abs(f1["sparsity"] - f2["sparsity"]),
        f"{prefix}_lbp_hist_dist": safe_cosine_hist(f1["hist"], f2["hist"]),
    }


In [ ]:
def compute_lbp_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)
    regions = create_face_regions(frame_std, bbox_scaled)
    lbp = compute_lbp(gray)

    face = lbp_region_features(lbp, regions["face"])
    border = lbp_region_features(lbp, regions["border"])
    bg = lbp_region_features(lbp, regions["background"])

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            if metric_name == "hist":
                continue
            features[f"lbp_{region_name}_{metric_name}"] = metric_value

    features.update(lbp_region_differences(face, bg, "face_bg"))
    features.update(lbp_region_differences(face, border, "face_border"))
    features.update(lbp_region_differences(border, bg, "border_bg"))

    return features, lbp


## Apenas para visualizar o LBP

In [ ]:
def normalize_lbp_for_display(lbp):
    lbp_norm = cv2.normalize(lbp, None, 0, 255, cv2.NORM_MINMAX)
    return lbp_norm.astype(np.uint8)


def debug_lbp_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    debug_frames = []

    print("\nGerando frames de debug...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])
        regions = create_face_regions(frame_std, bbox_scaled)

        features, lbp = compute_lbp_metrics(frame, bbox)
        lbp_vis = normalize_lbp_for_display(lbp)
        lbp_vis = cv2.cvtColor(lbp_vis, cv2.COLOR_GRAY2BGR)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face Entropy: {features.get('lbp_face_entropy_norm', 0):.3f}",
            f"Face Uniform: {features.get('lbp_face_uniformity', 0):.3f}",
            f"Face-BG HistDist: {features.get('face_bg_lbp_hist_dist', 0):.3f}",
            f"Face-Border HistDist: {features.get('face_border_lbp_hist_dist', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(lbp_vis, cv2.COLOR_BGR2RGB),
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nDebug pronto. Renderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")
    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(fig, update, frames=len(debug_frames), interval=interval, blit=True)

    plt.close(fig)
    mpl.rcParams["animation.embed_limit"] = 500000000
    display(HTML(ani.to_jshtml()))


In [10]:
#debug_lbp_video(df_true['video_path'].iloc[0], max_frames=300)

In [11]:
#debug_lbp_video(df_fake['video_path'].iloc[2], max_frames=300)

# Sobel

In [12]:
def compute_sobel(gray):

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    mag = cv2.magnitude(gx, gy)
    angle = cv2.phase(gx, gy, angleInDegrees=False)

    return mag, angle

In [ ]:
def sobel_region_features(mag, angle, mask, n_bins=12):
    mag_vals = mag[mask == 1].astype(np.float32)
    ang_vals = angle[mask == 1].astype(np.float32)

    if len(mag_vals) < 20:
        return None

    hist, _ = np.histogram(ang_vals, bins=n_bins, range=(0, 2 * np.pi), density=False)
    hist = hist.astype(np.float64)
    hist /= (hist.sum() + 1e-12)

    entropy = float(-np.sum(hist * np.log(hist + 1e-12)) / np.log(n_bins))

    weights = mag_vals / (np.sum(mag_vals) + 1e-12)
    mean_sin = float(np.sum(weights * np.sin(ang_vals)))
    mean_cos = float(np.sum(weights * np.cos(ang_vals)))
    coherence = float(np.sqrt(mean_sin ** 2 + mean_cos ** 2))

    energy = float(np.mean(mag_vals))
    std = float(np.std(mag_vals))
    p95 = float(np.percentile(mag_vals, 95))

    return {
        "hist": hist,
        "entropy_norm": entropy,
        "coherence": coherence,
        "energy": energy,
        "std": std,
        "p95": p95,
    }


In [ ]:
def sobel_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_sobel_entropy_norm_diff": abs(f1["entropy_norm"] - f2["entropy_norm"]),
        f"{prefix}_sobel_coherence_diff": abs(f1["coherence"] - f2["coherence"]),
        f"{prefix}_sobel_energy_diff": abs(f1["energy"] - f2["energy"]),
        f"{prefix}_sobel_std_diff": abs(f1["std"] - f2["std"]),
        f"{prefix}_sobel_p95_diff": abs(f1["p95"] - f2["p95"]),
        f"{prefix}_sobel_hist_dist": safe_cosine_hist(f1["hist"], f2["hist"]),
    }


In [ ]:
def compute_sobel_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)
    regions = create_face_regions(frame_std, bbox_scaled)
    mag, angle = compute_sobel(gray)

    face = sobel_region_features(mag, angle, regions["face"])
    border = sobel_region_features(mag, angle, regions["border"])
    bg = sobel_region_features(mag, angle, regions["background"])

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            if metric_name == "hist":
                continue
            features[f"sobel_{region_name}_{metric_name}"] = metric_value

    features.update(sobel_region_differences(face, bg, "face_bg"))
    features.update(sobel_region_differences(face, border, "face_border"))
    features.update(sobel_region_differences(border, bg, "border_bg"))

    return features, mag, angle


## Visualização para debug


In [ ]:
def sobel_visual(mag, angle):
    mag_vis = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    angle_vis = (angle / (2 * np.pi) * 255).astype(np.uint8)
    angle_vis = cv2.applyColorMap(angle_vis, cv2.COLORMAP_HSV)
    return mag_vis, angle_vis


def debug_sobel_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    debug_frames = []

    print("\nProcessando Sobel...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])
        regions = create_face_regions(frame_std, bbox_scaled)

        features, mag, angle = compute_sobel_metrics(frame, bbox)
        mag_vis, angle_vis = sobel_visual(mag, angle)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face Entropy: {features.get('sobel_face_entropy_norm', 0):.3f}",
            f"Face Coherence: {features.get('sobel_face_coherence', 0):.3f}",
            f"Face-BG HistDist: {features.get('face_bg_sobel_hist_dist', 0):.3f}",
            f"Face-BG EnergyDiff: {features.get('face_bg_sobel_energy_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(mag_vis, cv2.COLOR_GRAY2RGB),
            cv2.cvtColor(angle_vis, cv2.COLOR_BGR2RGB),
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.axis("off")
    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(fig, update, frames=len(debug_frames), interval=interval, blit=True)

    plt.close(fig)
    display(HTML(ani.to_jshtml()))


In [17]:
# debug_sobel_video(df_true['video_path'].iloc[0], max_frames=300)

In [18]:
# debug_sobel_video(df_fake['video_path'].iloc[1], max_frames=200)

# Laplace

In [19]:
def compute_laplacian(gray):
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    return lap

In [ ]:
from scipy.stats import kurtosis


def laplacian_region_features(lap, mask, n_bins=40):
    values = lap[mask == 1].astype(np.float32)

    if len(values) < 20:
        return None

    energy = float(np.mean(values ** 2))
    variance = float(np.var(values))
    mean_abs = float(np.mean(np.abs(values)))
    p95_abs = float(np.percentile(np.abs(values), 95))
    std = float(np.sqrt(variance))

    if std < 1e-6:
        values_norm = np.zeros_like(values)
        kurt = 0.0
    else:
        values_norm = (values - np.mean(values)) / std
        kurt = float(kurtosis(values_norm, fisher=True, bias=False))

    hist, _ = np.histogram(values_norm, bins=n_bins, range=(-5, 5), density=False)
    hist = hist.astype(np.float64)
    hist /= (hist.sum() + 1e-12)

    entropy = float(-np.sum(hist * np.log(hist + 1e-12)) / np.log(n_bins))
    sparsity = float(np.mean(np.abs(values_norm) > 1.0))

    return {
        "hist": hist,
        "energy": energy,
        "variance": variance,
        "mean_abs": mean_abs,
        "p95_abs": p95_abs,
        "entropy_norm": entropy,
        "kurtosis": kurt,
        "sparsity": sparsity,
    }


In [ ]:
def laplacian_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_lap_energy_diff": abs(f1["energy"] - f2["energy"]),
        f"{prefix}_lap_variance_diff": abs(f1["variance"] - f2["variance"]),
        f"{prefix}_lap_mean_abs_diff": abs(f1["mean_abs"] - f2["mean_abs"]),
        f"{prefix}_lap_p95_abs_diff": abs(f1["p95_abs"] - f2["p95_abs"]),
        f"{prefix}_lap_entropy_norm_diff": abs(f1["entropy_norm"] - f2["entropy_norm"]),
        f"{prefix}_lap_kurtosis_diff": abs(f1["kurtosis"] - f2["kurtosis"]),
        f"{prefix}_lap_sparsity_diff": abs(f1["sparsity"] - f2["sparsity"]),
        f"{prefix}_lap_hist_dist": safe_cosine_hist(f1["hist"], f2["hist"]),
    }


In [ ]:
def compute_laplacian_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)
    regions = create_face_regions(frame_std, bbox_scaled)
    lap = compute_laplacian(gray)

    face = laplacian_region_features(lap, regions["face"])
    border = laplacian_region_features(lap, regions["border"])
    bg = laplacian_region_features(lap, regions["background"])

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            if metric_name == "hist":
                continue
            features[f"lap_{region_name}_{metric_name}"] = metric_value

    features.update(laplacian_region_differences(face, bg, "face_bg"))
    features.update(laplacian_region_differences(face, border, "face_border"))
    features.update(laplacian_region_differences(border, bg, "border_bg"))

    return features, lap


## Visualização para debugging

In [ ]:
def laplacian_visual(lap):
    lap_abs = np.abs(lap)
    lap_vis = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    return lap_vis.astype(np.uint8)


def debug_laplacian_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    debug_frames = []

    print("\nProcessando Laplacian...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])
        regions = create_face_regions(frame_std, bbox_scaled)

        features, lap = compute_laplacian_metrics(frame, bbox)
        lap_vis = laplacian_visual(lap)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face Energy: {features.get('lap_face_energy', 0):.3f}",
            f"Face Kurtosis: {features.get('lap_face_kurtosis', 0):.3f}",
            f"Face-BG HistDist: {features.get('face_bg_lap_hist_dist', 0):.3f}",
            f"Face-BG KurtDiff: {features.get('face_bg_lap_kurtosis_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(lap_vis, cv2.COLOR_GRAY2RGB),
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")
    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(fig, update, frames=len(debug_frames), interval=interval, blit=True)

    plt.close(fig)
    display(HTML(ani.to_jshtml()))


In [24]:
# debug_laplacian_video(df_true['video_path'].iloc[1], max_frames=300)

In [25]:
# debug_laplacian_video(df_fake['video_path'].iloc[0], max_frames=300)

# Resumo de metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    all_features = []

    for i, frame in enumerate(frames):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        bbox = metadata[metadata_idx]["bbox"]

        features_lbp, _ = compute_lbp_metrics(frame, bbox)
        features_sobel, _, _ = compute_sobel_metrics(frame, bbox)
        features_lap, _ = compute_laplacian_metrics(frame, bbox)

        features = {**features_lbp, **features_sobel, **features_lap}
        features["frame"] = frame_idx
        features["metadata_idx"] = metadata_idx

        all_features.append(features)

    metrics_df = pd.DataFrame(all_features)
    metrics_df.insert(0, "video_name", video_name)

    if label is not None:
        metrics_df.insert(1, "label", label)

    return metrics_df


def metadata_exists(video_path):
    return os.path.exists(metadata_path_for_video(video_path))


def aggregate_video_metrics(frame_metrics):
    metric_cols = [
        col for col in frame_metrics.columns
        if col.startswith("lbp_") or "_lbp_" in col or col.startswith("sobel_") or "_sobel_" in col or col.startswith("lap_") or "_lap_" in col
    ]

    agg = frame_metrics[metric_cols].agg(["mean", "std", "median"])
    values = {}

    for metric in metric_cols:
        values[f"{metric}_mean"] = agg.loc["mean", metric]
        values[f"{metric}_std"] = agg.loc["std", metric]
        values[f"{metric}_median"] = agg.loc["median", metric]

    return values


def auc_from_scores(labels, scores, positive_label="Fake"):
    valid = [(label, score) for label, score in zip(labels, scores) if pd.notna(score)]
    pos = [score for label, score in valid if label == positive_label]
    neg = [score for label, score in valid if label != positive_label]

    if len(pos) == 0 or len(neg) == 0:
        return np.nan

    wins = 0
    ties = 0

    for pos_score in pos:
        for neg_score in neg:
            wins += pos_score > neg_score
            ties += pos_score == neg_score

    return (wins + 0.5 * ties) / (len(pos) * len(neg))


def cohen_d_by_label(values, labels, positive_label="Fake"):
    values = pd.Series(values)
    labels = pd.Series(labels)

    pos = values[labels == positive_label].dropna().astype(float)
    neg = values[labels != positive_label].dropna().astype(float)

    if len(pos) < 2 or len(neg) < 2:
        return np.nan

    pooled = np.sqrt((pos.var(ddof=1) + neg.var(ddof=1)) / 2)

    if pooled == 0 or pd.isna(pooled):
        return np.nan

    return (pos.mean() - neg.mean()) / pooled


def evaluate_group_a(max_frames=120):
    rows = []
    available_df = df[df['video_path'].apply(metadata_exists)].reset_index(drop=True)

    for _, row in available_df.iterrows():
        frame_metrics = all_metrics(
            row['video_path'],
            max_frames=max_frames,
            label=row['Video Ground Truth'],
        )

        if frame_metrics.empty:
            continue

        video_row = aggregate_video_metrics(frame_metrics)
        video_row["video_name"] = os.path.basename(row['video_path'])
        video_row["label"] = row['Video Ground Truth']
        video_row["n_frames"] = len(frame_metrics)
        rows.append(video_row)

    video_metrics = pd.DataFrame(rows)

    metric_cols = [
        col for col in video_metrics.columns
        if (col.startswith("lbp_") or "_lbp_" in col or col.startswith("sobel_") or "_sobel_" in col or col.startswith("lap_") or "_lap_" in col) and col.endswith("_mean")
    ]

    report_rows = []
    for metric in metric_cols:
        auc = auc_from_scores(video_metrics['label'], video_metrics[metric])
        auc_abs = max(auc, 1 - auc) if pd.notna(auc) else np.nan
        d = cohen_d_by_label(video_metrics[metric], video_metrics['label'])

        report_rows.append({
            "metric": metric,
            "auc_fake": auc,
            "auc_abs": auc_abs,
            "cohen_d_fake_minus_real": d,
            "real_mean": video_metrics.loc[video_metrics['label'] == "Real", metric].mean(),
            "fake_mean": video_metrics.loc[video_metrics['label'] == "Fake", metric].mean(),
        })

    report = pd.DataFrame(report_rows)

    if not report.empty:
        report = report.sort_values(
            ["auc_abs", "cohen_d_fake_minus_real"],
            ascending=[False, False],
            key=lambda s: s.abs() if s.name == "cohen_d_fake_minus_real" else s,
        ).reset_index(drop=True)

    print("\nVídeos avaliados:")
    print(video_metrics['label'].value_counts())

    return video_metrics, report


## Avaliação discriminativa

Executa a extração apenas nos vídeos com metadata disponível e resume uma linha por vídeo. Use `report.head(20)` para ver quais sinais de textura e borda separam melhor Real/Fake nesta amostra.

In [ ]:
# Rodada rapida de validacao. Aumente max_frames para estabilizar os resultados.
video_metrics, report = evaluate_group_a(max_frames=120)
display(report.head(20))


In [34]:
df.columns

Index(['video_name', 'lbp_face_entropy', 'lbp_face_uniformity',
       'lbp_border_entropy', 'lbp_bg_entropy', 'face_bg_entropy_diff',
       'face_bg_uniformity_diff', 'face_bg_sparsity_diff', 'face_bg_hist_dist',
       'face_border_entropy_diff', 'face_border_uniformity_diff',
       'face_border_sparsity_diff', 'face_border_hist_dist',
       'border_bg_entropy_diff', 'border_bg_uniformity_diff',
       'border_bg_sparsity_diff', 'border_bg_hist_dist', 'sobel_face_entropy',
       'sobel_face_coherence', 'face_bg_coherence_diff', 'face_bg_energy_diff',
       'face_border_coherence_diff', 'face_border_energy_diff',
       'border_bg_coherence_diff', 'border_bg_energy_diff', 'lap_face_energy',
       'lap_face_kurtosis', 'face_bg_kurtosis_diff',
       'face_border_kurtosis_diff', 'border_bg_kurtosis_diff', 'frame',
       'label'],
      dtype='str')